### Import packages

In [ ]:
import functions as fun
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import dalex as dx
import lime
import lime.lime_tabular

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.inspection import permutation_importance

### Load model to be explain

In [ ]:
pickle_in = open("..\models\pickle\MEPS_xgb_model_final_v2_cat_vars_str.pickle", "rb")
reg_xgb = pickle.load(pickle_in)

### Read data

In [ ]:
path = "..\data\MEPS_data_preprocessed"
X_train, y_train = fun.read_x_y(path + "_train.csv", "HEALTHEXP")
X_test, y_test = fun.read_x_y(path + "_test.csv", "HEALTHEXP")

In [ ]:
raw_test_data = pd.read_csv(path + "_test.csv")
X_test_raw = raw_test_data.drop("HEALTHEXP", axis = 1)
y_test_raw = raw_test_data["HEALTHEXP"]

In [ ]:
categorical_features = fun.get_categorical_features(X_train)

In [ ]:
for f in categorical_features:
    X_test[f] = X_test[f].astype("str")
    X_train[f] = X_train[f].astype("str")

### Find and explore observation with the worst prediction

In [ ]:
y_pred_test = reg_xgb.predict(X_test)
obs_idx = fun.find_nth_obs_idx(y_test, y_pred_test, 0)

In [ ]:
# Real value (target transformed with log base 3)
y_test[obs_idx]

In [ ]:
# Real value (raw)
y_test_raw[obs_idx]

In [ ]:
# Predicted value
y_pred_test[obs_idx]

In [ ]:
# Observation
obs = X_test.iloc[[obs_idx], :]
obs

**Meaning of variables and their values:** <br/>
* REGION = 3.0 	 - 	 south
* AGE31X = 76.0 	 - 	 16 years old
* GENDER = 1.0 	 - 	 male
* RACE3 = 0.0 	 - 	 
* MARRY31X = 2.0 	 - 	 widowed
* EDRECODE = 13.0 	 - 	 GED or high school degree
* FTSTU31X = -1.0 	 - 	 student status - inapplicable
* ACTDTY31 = 4.0 	 - 	 military full-time active duty - over 59 - inapplicable
* HONRDC31 = 2.0 	 - 	 not honorably discharged from military
* RTHLTH31 = 5.0 	 - 	 perceived health status - poor
* MNHLTH31 = 4.0 	 - 	 perceived mental health status - fair
* HIBPDX = 1.0 	 - 	 high blood pressure diagnosed
* CHDDX = 1.0 	 - 	 coronary heart disease diagnosed
* ANGIDX = 2.0 	 - 	 angina wasn't diagnosed
* MIDX = 2.0 	 - 	 heart attack wasn't diagnosed
* OHRTDX = 2.0 	 - 	 any other heart diseases weren't diagnosed
* STRKDX = 2.0 	 - 	 stroke wasn't diagnosed
* EMPHDX = 1.0 	 - 	 emphysema diagnosed
* CHBRON31 = 1.0 	 - 	 chronic bronchitis diagnosed
* CHOLDX = 1.0 	 - 	 high cholesterol diagnosed
* CANCERDX = 1.0 	 - 	 cancer diagnosed
* DIABDX = 2.0 	 - 	 diabetes wasn't diagnosed
* JTPAIN31 = 1.0 	 - 	 joint pain last 12 months diagnosed
* ARTHDX = 1.0 	 - 	 arthritis diagnosed
* ARTHTYPE = 1.0 	 - 	 type of arthritis - rheumatoid
* ASTHDX = 2.0 	 - 	 asthma wasn't diagnosed
* ADHDADDX = -1.0 	 - 	 ADHD or ADD diagnosis - inapplicable
* PREGNT31 = -1.0 	 - 	 pregnant - inapplicable
* WLKLIM31 = 1.0 	 - 	 has limitation in physical functioning
* ACTLIM31 = 1.0 	 - 	 has any other limitation work/house work/school
* SOCLIM31 = 1.0 	 - 	 has social limitation
* COGLIM31 = 1.0 	 - 	 has cognitive limitation
* DFHEAR42 = 2.0 	 - 	 hasn't serious difficulty hearing
* DFSEE42 = 1.0 	 - 	 has serious difficulty see or wears glasses
* ADSMOK42 = -1.0 	 - 	 doesn't smoke
* PCS42 = -1.0 	 - 	 saq:phy component summry sf-12v2 imputed - inapplicable
* MCS42 = -1.0 	 - 	 mnt component summry sf-12v2 imputed - inapplicable
* K6SUM42 = -1.0 	 - 	 overall rating of feelings - inapplicable (last 30 days)
* PHQ242 = -1.0 	 - 	 overall rating of feelings - inapplicable (last 2 weeks)
* EMPST31 = 4.0 	 - 	 employment status - ?
* POVCAT15 = 5.0 	 - 	 family income as % of poverty line - high income
* INSCOV15 = 2.0 	 - 	 health insurance coverage indicator 2015 - public only
* INCOME_M = 12345.0 - 	 person total income = 12345.0 

### Explanations

#### 1) Break Down / Shap

In [ ]:
import dalex as dx

In [ ]:
X_train

In [ ]:
for col in ['RTHLTH31', 'ADSMOK42', 'INSCOV15','ACTDTY31','MNHLTH31','PREGNT31']:
    X_train[col] =  X_train[col].astype(str)

In [ ]:
exp = dx.Explainer(reg_xgb, X_train, y_train, label = "MEPS XGB Pipeline")

In [ ]:
X_train

In [ ]:
X_train.info()

In [ ]:
pdp_cat = exp.model_profile(type = 'partial', variable_type='categorical',
                            variables = ["ADSMOK42","MNHLTH31","PREGNT31"])
pdp_cat.result['_label_'] = 'pdp'

ale_cat = exp.model_profile(type = 'accumulated', variable_type='categorical', 
                            variables = ["ADSMOK42","MNHLTH31",'PREGNT31'])
ale_cat.result['_label_'] = 'ale'



In [ ]:
fig5 = ale_cat.plot(pdp_cat, size = 4, facet_ncol=1)
# fig5 = ale_cat.plot( size = 4, facet_ncol=1)
# fig5.write_image("06_dataset_ADSMOK42_pdp_ale.png", scale = 2)

In [ ]:
pdp_cat = exp.model_profile(type = 'partial', variable_type='numerical',
                            variables = ["AGE31X"])
pdp_cat.result['_label_'] = 'pdp'

ale_cat = exp.model_profile(type = 'accumulated', variable_type='numerical', 
                            variables = ["AGE31X"])
ale_cat.result['_label_'] = 'ale'

In [ ]:
# fig = go.Figure()
fig = ale_cat.plot(pdp_cat, size = 4, facet_ncol=1)

In [ ]:
X_train.columns

In [ ]:
X_train[X_train['GENDER']=="1.0"]['PREGNT31'].value_counts()

In [ ]:
X_train_male = X_train[X_train['GENDER']!="1.0"]
y_train_male = y_train[X_train_male.index]

In [ ]:
X_train_female = X_train[X_train['GENDER']=="1.0"]
y_train_female = y_train[X_train_female.index]

In [ ]:
# explainer for Sex1 and sex2
exp_male = dx.Explainer(reg_xgb, X_train_male, y_train_male, label = "MEPS XGB Pipeline")
exp_female = dx.Explainer(reg_xgb, X_train_female, y_train_female, label = "MEPS XGB Pipeline")

In [ ]:
pdp_cat_male = exp_male.model_profile(type = 'partial', variable_type='numerical',
                            variables = ["AGE31X"])
pdp_cat_male.result['_label_'] = 'pdp_male'

ale_cat_male = exp_male.model_profile(type = 'accumulated', variable_type='numerical', 
                            variables = ["AGE31X"])
ale_cat_male.result['_label_'] = 'ale_male'

In [ ]:
pdp_cat_female = exp_female.model_profile(type = 'partial', variable_type='numerical',
                            variables = ["AGE31X"])
pdp_cat_female.result['_label_'] = 'pdp_female'

ale_cat_female = exp_female.model_profile(type = 'accumulated', variable_type='numerical', 
                            variables = ["AGE31X"])
ale_cat_female.result['_label_'] = 'ale_female'

In [ ]:
fig = pdp_cat_male.plot(ale_cat_male, size = 4, facet_ncol=1)

In [ ]:
fig = pdp_cat_female.plot(ale_cat_female, size = 4, facet_ncol=1)

In [ ]:
fig = ale_cat_female.plot(ale_cat_male, size = 4, facet_ncol=1)

In [ ]:
fig = pdp_cat_female.plot(pdp_cat_male, size = 4, facet_ncol=1)

In [ ]:
pdp_cat = exp.model_profile(type = 'partial', variable_type='categorical',
                            variables = ['RTHLTH31'])
pdp_cat.result['_label_'] = 'pdp'

ale_cat = exp.model_profile(type = 'accumulated', variable_type='categorical', 
                            variables = ['RTHLTH31'])
ale_cat.result['_label_'] = 'ale'

In [ ]:
fig2 = ale_cat.plot(pdp_cat, size = 4, facet_ncol=1)

In [ ]:
fig2.write_image("06_RTHLTH31_pdp_ale.png", scale = 2)

In [ ]:
pdp_cat = exp.model_profile(type = 'partial', variable_type='numerical',
                            variables = ["PCS42"])
pdp_cat.result['_label_'] = 'pdp'

ale_cat = exp.model_profile(type = 'accumulated', variable_type='numerical', 
                            variables = ["PCS42"])
ale_cat.result['_label_'] = 'ale'

In [ ]:
fig3 = ale_cat.plot(pdp_cat, size = 4, facet_ncol=1)
fig3.write_image("06_PCS42_pdp_ale.png", scale = 2)

In [ ]:
pdp_cat = exp.model_profile(type = 'partial', variable_type='categorical',
                            variables = ["MNHLTH31"])
pdp_cat.result['_label_'] = 'pdp'

ale_cat = exp.model_profile(type = 'accumulated', variable_type='categorical', 
                            variables = ["MNHLTH31"])
ale_cat.result['_label_'] = 'ale'

In [ ]:
fig4 = ale_cat.plot(pdp_cat, size = 4, facet_ncol=1)
fig4.write_image("06_MNHLTH31_pdp_ale.png", scale = 2)

In [ ]:
X_train['ADSMOK42']